## Phase 2A. PCA 기반 노이즈 제거 및 성능 비교

### 목표
- 7개 Feature 간 중복/상관관계 제거
- PCA로 핵심 주성분만 추출
- Phase 1 Random Forest와 성능 비교

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from src.data_loader import download_all
from src.features   import process_all, build_combined_dataset
from src.model      import split_data, train_model, evaluate_model, FEATURE_COLS

# 데이터 로드
stock_data = download_all(save_csv=False)
processed  = process_all(stock_data, save_csv=False)
combined   = build_combined_dataset(processed)

print("데이터 로드 완료")
print(f"Feature 목록: {FEATURE_COLS}")

c:\Users\user\miniconda3\envs\semiconductor\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[NVDA] 다운로드 중...


## 1. Feature 간 상관관계 확인

PCA 적용 전, Feature들이 실제로 중복된 정보를 담고 있는지 확인한다.
상관계수가 높을수록 중복 정보가 많다는 뜻이다.

In [ ]:
# Feature 간 상관관계 히트맵
X = combined[FEATURE_COLS]

plt.figure(figsize=(10, 8))
corr_matrix = X.corr()

sns.heatmap(
    corr_matrix,
    annot    = True,
    fmt      = ".2f",
    cmap     = "RdYlGn",
    center   = 0,
    linewidths = 0.5,
    square   = True
)
plt.title("Feature 간 상관관계 히트맵\n(절댓값이 클수록 중복 정보 많음)",
          fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/figures/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# 상관계수 0.5 이상인 쌍 출력
print("\n상관계수 0.5 이상인 Feature 쌍:")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) >= 0.5:
            print(f"  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {val:.3f}")

### 2. PCA 적용

#### 순서
1. StandardScaler로 Feature 정규화 (필수!)
2. PCA로 주성분 추출
3. 설명력(분산 비율) 확인 → 주성분 개수 결정
4. Random Forest 재학습
5. Phase 1과 성능 비교

#### 왜 정규화가 필요한가?
- RSI는 0~100 범위
- Volume_ratio는 0~5 범위
- 범위가 다르면 PCA가 큰 숫자에 치우침
- StandardScaler로 평균 0, 표준편차 1로 통일

In [ ]:
# ── 1. 학습/테스트 분리 ──────────────────────────────────────
X_train, X_test, y_train, y_test = split_data(combined)

# ── 2. 정규화 (StandardScaler) ───────────────────────────────
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # 학습 데이터로 fit
X_test_scaled  = scaler.transform(X_test)        # 테스트는 transform만

# ── 3. PCA 전체 주성분 추출 (설명력 확인용) ──────────────────
pca_full = PCA()
pca_full.fit(X_train_scaled)

# 누적 설명력 계산
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

print("주성분별 설명력:")
for i, (var, cum) in enumerate(zip(
    pca_full.explained_variance_ratio_, cumulative_variance)):
    print(f"  PC{i+1}: {var*100:.1f}%  (누적: {cum*100:.1f}%)")

# ── 4. 누적 설명력 그래프 ────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(cumulative_variance)+1),
        pca_full.explained_variance_ratio_ * 100,
        alpha=0.7, color="#3498db", label="개별 설명력")
plt.plot(range(1, len(cumulative_variance)+1),
         cumulative_variance * 100,
         "ro-", linewidth=2, label="누적 설명력")
plt.axhline(y=80, color="gray", linestyle="--", alpha=0.7, label="80% 기준선")
plt.axhline(y=90, color="black", linestyle="--", alpha=0.7, label="90% 기준선")

plt.title("PCA 주성분별 설명력", fontsize=14, fontweight="bold")
plt.xlabel("주성분 (PC)")
plt.ylabel("설명력 (%)")
plt.xticks(range(1, len(cumulative_variance)+1))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/figures/pca_variance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 주성분 개수 결정 ─────────────────────────────────────────
# 누적 설명력 80% 이상을 유지하는 최소 주성분 개수 선택
n_components = np.argmax(cumulative_variance >= 0.80) + 1
print(f"선택된 주성분 개수: {n_components}개 (누적 설명력 80% 이상)")

# ── PCA 변환 ─────────────────────────────────────────────────
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"\n변환 전 Shape: {X_train.shape}")
print(f"변환 후 Shape: {X_train_pca.shape}")

# ── PCA 적용 후 Random Forest 학습 ───────────────────────────
X_train_pca_df = pd.DataFrame(
    X_train_pca,
    columns=[f"PC{i+1}" for i in range(n_components)]
)
X_test_pca_df = pd.DataFrame(
    X_test_pca,
    columns=[f"PC{i+1}" for i in range(n_components)]
)

model_pca   = train_model(X_train_pca_df, y_train)
results_pca = evaluate_model(model_pca, X_test_pca_df, y_test)

In [ ]:
# ── Phase 1 결과 (비교 기준) ─────────────────────────────────
model_phase1   = train_model(X_train, y_train)
results_phase1 = evaluate_model(model_phase1, X_test, y_test)

# ── 성능 비교 표 ─────────────────────────────────────────────
comparison = pd.DataFrame({
    "모델"      : ["Phase 1 (RF, 7 Features)", 
                   f"Phase 2A (RF + PCA, {n_components} 주성분)"],
    "Accuracy" : [results_phase1["accuracy"], results_pca["accuracy"]],
    "AUC"      : [results_phase1["auc"],      results_pca["auc"]],
    "Feature 수": [7, n_components]
})

print("\nPhase 1 vs Phase 2A 성능 비교")
display(comparison.round(4))

# ── 시각화 ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics  = ["Accuracy", "AUC"]
phase1   = [results_phase1["accuracy"], results_phase1["auc"]]
phase2a  = [results_pca["accuracy"],    results_pca["auc"]]
x        = np.arange(len(metrics))
width    = 0.35

for ax, metric, p1, p2 in zip(axes, metrics, phase1, phase2a):
    bars1 = ax.bar(0, p1, width, label="Phase 1 (7 Features)", color="#3498db")
    bars2 = ax.bar(1, p2, width, label=f"Phase 2A ({n_components} 주성분)", color="#e74c3c")
    ax.set_title(f"{metric} 비교", fontsize=13, fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Phase 1", "Phase 2A"])
    ax.set_ylim(0.48, 0.60)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    ax.text(0, p1 + 0.001, f"{p1:.4f}", ha="center", fontsize=11, fontweight="bold")
    ax.text(1, p2 + 0.001, f"{p2:.4f}", ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Phase 1 vs Phase 2A 성능 비교", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/figures/phase1_vs_phase2a.png", dpi=150, bbox_inches="tight")
plt.show()